# Phase-Amplitude

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Simulation parameters
sfreq = 1000             # Sampling frequency in Hz
duration = 10           # Duration in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# 1. Generate low-frequency (theta) signal
lf_freq = 4  # Hz
lf_signal = np.sin(2 * np.pi * lf_freq * times) # predictor for low-frequency phase (e.g, speech rhythm)

# Extract phase
lf_phase = np.angle(hilbert(lf_signal))

# 2. Generate high-frequency (gamma) signal
hf_freq = 80  # Hz
hf_carrier = np.sin(2 * np.pi * hf_freq * times)

# --- Modulate HF amplitude by LF phase (PAC) ---
modulation_index = 0.8  # --> amplitude envelope
hf_envelope = 1 + modulation_index * np.cos(lf_phase)  # target/response envelope modulated by LF phase
hf_pac_signal = hf_envelope * hf_carrier # phase-to-amplitude mapping

# --- Simulated EEG = PAC signal + Gaussian noise ---
noise = np.random.randn(n_samples) * 0.5
sim_eeg = hf_pac_signal + noise # simulated EEG trace that contains known PAC

# Time lags
tmin = -0.1   # in seconds (−100 ms)
tmax = 0.4    # in seconds (+400 ms)
lags = np.arange(int(tmin * sfreq), int(tmax * sfreq) + 1)
n_lags = len(lags)

# --- Create time-lagged predictor matrix ---
# Each column = predictor shifted by a time lag
def make_lagged_matrix(predictor, lags):
    padded = np.pad(predictor, (np.max(np.abs(lags)),), mode='constant')
    X_lagged = np.stack([
        padded[np.max(np.abs(lags)) + lag : np.max(np.abs(lags)) + lag + len(predictor)]
        for lag in lags
    ], axis=1)
    return X_lagged

X_lagged = make_lagged_matrix(lf_signal, lags)

# --- Standardize predictors and response ---
X_std = StandardScaler().fit_transform(X_lagged)
Y_std = StandardScaler().fit_transform(sim_eeg.reshape(-1, 1)).ravel()

# --- Fit mTRF using ridge regression ---
ridge = Ridge(alpha=1.0)  # alpha = regularization strength
ridge.fit(X_std, Y_std)

# --- Extract TRF weights ---
trf_weights = ridge.coef_

# --- Plot TRF ---
plt.figure(figsize=(6, 4))
plt.plot(lags * 1000, trf_weights, label='mTRF (PAC)')
plt.axhline(0, color='gray', linestyle='--')
plt.title('Phase-Amplitude Coupling')
plt.xlabel('Lag (ms)')
plt.ylabel('TRF weight (a.u.)')
plt.legend()
plt.tight_layout()
plt.show()


# Simulated AAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000   # Sampling rate in Hz
duration = 10  # Duration of the signal in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier + fluctuations
lf_freq = 4 #low-frequency rhythm (e.g., theta band).
lf_signal = np.sin(2 * np.pi * lf_freq * times) #4 Hz sine wave — low-frequency carrier.
lf_amp = np.abs(hilbert(lf_signal))  # Use the Hilbert transform to extract the instantaneous amplitude (the envelope) of the LF signal

# --- Step 1: Generate slow amplitude modulator amp_mod(t) ---
amp_mod = np.zeros(n_samples)
for _ in range(5):  # sum of 5 random sinusoids < 1 Hz
    f_i = np.random.uniform(0.05, 0.5)  # random freq <= 0.5 Hz
    phi_i = np.random.uniform(0, 2 * np.pi)
    amp_mod += np.cos(2 * np.pi * f_i * times + phi_i)
amp_mod = (amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())  # normalize to [0,1]
# generate a slow-varying envelope amp_mod(t) by summing 5 random low-frequency sinusoids (< 1 Hz).
# Each component has a random frequency and random phase.
# then normalize the total signal to be in range [0, 1].
# amp_mod will control how much the LF envelope drives the HF envelope.

# --- Step 2: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(0, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())
#  a control envelope that is not related to the LF signal.
# represents independent background amplitude fluctuation in the gamma band.

# --- Step 3: Final high-frequency envelope ---
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * lf_amp
# blend hf_amp_ind and lf_amp using amp_mod as a mixing coefficient
# At time t, if amp_mod[t] is close to 0, then hf_envelope[t] ≈ hf_amp_ind[t].
# If amp_mod[t] is close to 1, then hf_envelope[t] ≈ lf_amp[t].
# This simulates intermittent PAC, rather than constant coupling — realistic EEG-inspired design.

# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times) # 80 Hz sine wave.
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples) #  this creates amplitude-modulated HF signal.
# Add Gaussian noise to simulate realistic EEG noise


# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color = 'lightgrey', label='Simulated PAC EEG')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='--')
plt.plot(times, lf_amp, label='lf_amp(t)', linestyle='--')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
PAC

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# Setup
sfreq = 1000   # Sampling rate in Hz
duration = 1  # Duration of the signal in seconds
n_samples = int(sfreq * duration)
times = np.arange(n_samples) / sfreq

# Low-freq carrier
lf_freq = 4 #low-frequency rhythm (e.g., theta band).
lf_signal = np.sin(2 * np.pi * lf_freq * times) #4 Hz sine wave — low-frequency carrier.
lf_phase = np.angle(hilbert(lf_signal))  # Use the Hilbert transform to extract the instantaneous amplitude (the envelope) of the LF signal

# --- Step 1: Generate slow amplitude modulator amp_mod(t) ---
amp_mod = np.zeros(n_samples)
for _ in range(5):  # sum of 5 random sinusoids < 1 Hz
    f_i = np.random.uniform(0.5, 2)  # random freq <= 0.5 Hz
    phi_i = np.random.uniform(-2 * np.pi, 2 * np.pi)
    a_i= np.random.uniform(1,5)
    amp_mod += np.cos(2 * np.pi * f_i * times + phi_i)* a_i
amp_mod =.25 + .75*(amp_mod - amp_mod.min()) / (amp_mod.max() - amp_mod.min())  # normalize to [0,1]
# generate a slow-varying envelope amp_mod(t) by summing 5 random low-frequency sinusoids (< 1 Hz).
# Each component has a random frequency and random phase.
# then normalize the total signal to be in range [0, 1].
# amp_mod will control how much the LF envelope drives the HF envelope.

# --- Step 2: Generate independent HF amplitude envelope hf_amp_ind(t) ---
hf_amp_ind = np.zeros(n_samples)
for _ in range(5):  # similar idea: unrelated slow rhythm
    f_i = np.random.uniform(0.05, 0.5)
    phi_i = np.random.uniform(-2*np.pi, 2 * np.pi)
    hf_amp_ind += np.cos(2 * np.pi * f_i * times + phi_i)
hf_amp_ind = (hf_amp_ind - hf_amp_ind.min()) / (hf_amp_ind.max() - hf_amp_ind.min())
#  a control envelope that is not related to the LF signal.
# represents independent background amplitude fluctuation in the gamma band.

# --- Step 3: Final high-frequency envelope ---
hf_amp_lf = np.exp((-(lf_phase)**2) / (np.pi**2 /2))
hf_envelope = (1 - amp_mod) * hf_amp_ind + amp_mod * hf_amp_lf
# blend hf_amp_ind and lf_amp using amp_mod as a mixing coefficient
# At time t, if amp_mod[t] is close to 0, then hf_envelope[t] ≈ hf_amp_ind[t].
# If amp_mod[t] is close to 1, then hf_envelope[t] ≈ lf_amp[t].
# This simulates intermittent PAC, rather than constant coupling — realistic EEG-inspired design.

# HF carrier (gamma band)
hf_freq = 80
hf_carrier = np.sin(2 * np.pi * hf_freq * times) # 80 Hz sine wave.
hf_signal = hf_envelope * hf_carrier + 0.3 * np.random.randn(n_samples) #  this creates amplitude-modulated HF signal.
# Add Gaussian noise to simulate realistic EEG noise


# Plot
plt.figure(figsize=(10, 4))
plt.plot(times, hf_signal, color = 'lightgrey', label='Simulated PAC EEG')
plt.plot(times, lf_signal, label='lf_signal', color='red')
plt.plot(times, hf_envelope, label='PAC Envelope', alpha=0.7)
plt.plot(times, amp_mod, label='amp_mod(t)', linestyle='--')
plt.plot(times, lf_phase, label='lf_phase(t)', linestyle=':')
plt.title("Data-Inspired PAC Simulation (Weissbart-style)")
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.tight_layout()
plt.show()
